# DeepVision – XGBoost Models & Product Clustering

This notebook trains three machine learning models on the `amazon_cleaned_final_with_log_price.xls` dataset:

1. **XGBoost Regressor** to predict the log of the discounted price (`log_price`).
2. **XGBoost Classifier** to predict the price category (`price_category_num`).
3. **K-Means clustering** to discover product groups.

It also shows how to export the models for later use in a web application (e.g. your HTML interface).

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.cluster import KMeans

from xgboost import XGBRegressor, XGBClassifier
import joblib

pd.set_option('display.max_columns', None)

## 1. Load and inspect the data

In [4]:
# IMPORTANT: the file is actually a CSV saved with an .xls extension.
data_path = 'amazon_cleaned_final_with_log_price.xls'
df = pd.read_csv(data_path)
df.head()

,product_id,product_name,main_category,discounted_price_clean,actual_price_clean,discount_pct,discount_amount,rating_clean,rating_count_clean,price_category,popularity_score,log_price
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,399.0,1099.0,64.0,700.0,4.2,24269,Bas,42.407384,5.991465
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories,199.0,349.0,43.0,150.0,4.0,43994,Très bas,42.767325,5.298317
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories,199.0,1899.0,90.0,1700.0,3.9,7928,Très bas,35.015301,5.298317
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories,329.0,699.0,53.0,370.0,4.2,94363,Bas,48.110643,5.799093
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories,154.0,399.0,61.0,245.0,4.2,16905,Très bas,40.888780,5.043425


## 2. Prepare targets and feature columns

In [6]:
# Inspect columns
print(df.columns)

# Map price category (string) → numeric labels
# Spec: 0 = bas (incl. 'Très bas'), 1 = moyen, 2 = élevé
price_category_mapping = {
    'Très bas': 0,
    'Bas': 0,
    'Moyen': 1,
    'Élevé': 2
}

df['price_category_num'] = df['price_category'].map(price_category_mapping)
df['price_category_num'].value_counts(dropna=False)

Index(['product_id', 'product_name', 'main_category', 'discounted_price_clean',
       'actual_price_clean', 'discount_pct', 'discount_amount', 'rating_clean',
       'rating_count_clean', 'price_category', 'popularity_score',
       'log_price'],
      dtype='object')


price_category_num
2    637
0    576
1    252
Name: count, dtype: int64

In [7]:
# Define feature columns used by all models
feature_cols = [
    'main_category',
    'actual_price_clean',
    'discount_pct',
    'rating_clean',
    'rating_count_clean',
    'popularity_score'
]

X = df[feature_cols].copy()
y_reg = df['log_price']                    # Regression target
y_clf = df['price_category_num']           # Classification target

X.head()

,main_category,actual_price_clean,discount_pct,rating_clean,rating_count_clean,popularity_score
0,Computers&Accessories,1099.0,64.0,4.2,24269,42.407384
1,Computers&Accessories,349.0,43.0,4.0,43994,42.767325
2,Computers&Accessories,1899.0,90.0,3.9,7928,35.015301
3,Computers&Accessories,699.0,53.0,4.2,94363,48.110643
4,Computers&Accessories,399.0,61.0,4.2,16905,40.888780


## 3. Train / test split

In [9]:
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

X_train.shape, X_test.shape

((1172, 6), (293, 6))

## 4. Preprocessing (numeric + categorical)

In [11]:
from sklearn.impute import SimpleImputer

numeric_features = [
    'actual_price_clean',
    'discount_pct',
    'rating_clean',
    'rating_count_clean',
    'popularity_score'
]
categorical_features = ['main_category']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## 5. XGBoost Regressor for `log_price`

In [13]:
regressor = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', regressor)
])

reg_pipeline.fit(X_train, y_train_reg)

y_pred_reg = reg_pipeline.predict(X_test)
rmse = mean_squared_error(y_test_reg, y_pred_reg, squared=False)
r2 = r2_score(y_test_reg, y_pred_reg)
print(f'RMSE (log_price): {rmse:.4f}')
print(f'R² (log_price): {r2:.4f}')

RMSE (log_price): 0.1222
R² (log_price): 0.9933


C:\Users\nourm\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [14]:
# Save the regression pipeline
joblib.dump(reg_pipeline, 'xgb_reg_log_price.pkl')
print('Saved model → xgb_reg_log_price.pkl')

Saved model → xgb_reg_log_price.pkl


## 6. XGBoost Classifier for `price_category_num`

In [16]:
classifier = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)

clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', classifier)
])

clf_pipeline.fit(X_train, y_train_clf)

y_pred_clf = clf_pipeline.predict(X_test)
acc = accuracy_score(y_test_clf, y_pred_clf)
print(f'Accuracy (price_category_num): {acc:.4f}')
print('\nClassification report:')
print(classification_report(y_test_clf, y_pred_clf))

Accuracy (price_category_num): 0.9829

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       115
           1       0.93      0.98      0.95        51
           2       1.00      0.98      0.99       127

    accuracy                           0.98       293
   macro avg       0.97      0.98      0.98       293
weighted avg       0.98      0.98      0.98       293



In [17]:
# Save the classification pipeline
joblib.dump(clf_pipeline, 'xgb_clf_price_category.pkl')
print('Saved model → xgb_clf_price_category.pkl')

Saved model → xgb_clf_price_category.pkl


## 7. K-Means clustering for product groups

In [23]:
# --- FIX: Handle NaN values before clustering ---

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Numeric features used for clustering
cluster_features = [
    'actual_price_clean',
    'discount_pct',
    'rating_clean',
    'rating_count_clean',
    'popularity_score',
    'log_price'
]

# Extract features
X_cluster = df[cluster_features].copy()

# 1. Impute missing values (median is best for numeric)
imputer = SimpleImputer(strategy='median')
X_cluster_imputed = imputer.fit_transform(X_cluster)

# 2. Scale the imputed values
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster_imputed)

# 3. Train k-means on clean inputs
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster_scaled)

df['cluster'].value_counts()


cluster
0    562
3    429
2    290
1    142
4     42
Name: count, dtype: int64

In [25]:
# Save clustering objects (scaler + kmeans)
cluster_bundle = {
    'scaler': scaler_cluster,
    'kmeans': kmeans,
    'features': cluster_features
}
joblib.dump(cluster_bundle, 'kmeans_products.pkl')
print('Saved clustering bundle → kmeans_products.pkl')

Saved clustering bundle → kmeans_products.pkl


In [27]:
# Simple cluster profiling: mean values per cluster
cluster_profile = df.groupby('cluster')[cluster_features + ['price_category']].agg({
    'actual_price_clean': 'mean',
    'discount_pct': 'mean',
    'rating_clean': 'mean',
    'rating_count_clean': 'mean',
    'popularity_score': 'mean',
    'log_price': 'mean',
    'price_category': lambda x: x.value_counts().index[0]
})
cluster_profile

,actual_price_clean,discount_pct,rating_clean,rating_count_clean,popularity_score,log_price,price_category
cluster,,,,,,,
0,2387.831530,63.071174,4.193772,17148.759786,36.361008,6.194607,Bas
1,31966.535211,36.359155,4.197887,14039.281690,36.604827,9.755140,Élevé
2,2330.962069,57.279310,3.723448,2132.268966,22.985561,6.384568,Bas
3,3073.417156,24.836830,4.181776,12122.659674,36.216379,7.104626,Élevé
4,2410.166667,47.452381,4.161905,221814.690476,50.895593,6.744395,Élevé


## 8. Helper functions for a web API (single product prediction)

In [30]:
def build_input_dataframe(
    product_name: str,
    main_category: str,
    actual_price_clean: float,
    discount_pct: float,
    rating_clean: float,
    rating_count_clean: int,
    popularity_score: float,
):
    """Create a one-row DataFrame with the same columns as the training features.
    product_name is not used by the models but can be useful for logging/trace.
    """
    return pd.DataFrame({
        'product_name': [product_name],
        'main_category': [main_category],
        'actual_price_clean': [actual_price_clean],
        'discount_pct': [discount_pct],
        'rating_clean': [rating_clean],
        'rating_count_clean': [rating_count_clean],
        'popularity_score': [popularity_score],
    })[feature_cols]


def predict_for_product(
    reg_model_path='xgb_reg_log_price.pkl',
    clf_model_path='xgb_clf_price_category.pkl',
    cluster_bundle_path='kmeans_products.pkl',
    **product_kwargs
):
    """Load saved models and return predictions for a single product.

    Returns a dictionary with:
    - predicted_log_price
    - predicted_discounted_price (exp of log_price)
    - price_category_num and label
    - cluster index
    """
    # Build input
    x_df = build_input_dataframe(**product_kwargs)

    # Load models
    reg_model = joblib.load(reg_model_path)
    clf_model = joblib.load(clf_model_path)
    cluster_bundle = joblib.load(cluster_bundle_path)
    scaler_cluster = cluster_bundle['scaler']
    kmeans = cluster_bundle['kmeans']
    cluster_features_local = cluster_bundle['features']

    # Regression
    pred_log_price = float(reg_model.predict(x_df)[0])
    pred_discounted_price = float(np.exp(pred_log_price))

    # Classification
    pred_cat_num = int(clf_model.predict(x_df)[0])
    inv_price_category_mapping = {
        0: 'Bas',
        1: 'Moyen',
        2: 'Élevé'
    }
    pred_cat_label = inv_price_category_mapping.get(pred_cat_num, 'Inconnu')

    # Cluster prediction (using numeric features only)
    # Here we approximate/derive needed clustering features from the provided info.
    # For a real API, make sure you compute them exactly the same way as in training.
    cluster_input = pd.DataFrame({
        'actual_price_clean': [product_kwargs['actual_price_clean']],
        'discount_pct': [product_kwargs['discount_pct']],
        'rating_clean': [product_kwargs['rating_clean']],
        'rating_count_clean': [product_kwargs['rating_count_clean']],
        'popularity_score': [product_kwargs['popularity_score']],
        # We use the predicted log price as an approximation
        'log_price': [pred_log_price]
    })[cluster_features]

    cluster_scaled = scaler_cluster.transform(cluster_input)
    cluster_idx = int(kmeans.predict(cluster_scaled)[0])

    return {
        'predicted_log_price': pred_log_price,
        'predicted_discounted_price': pred_discounted_price,
        'price_category_num': pred_cat_num,
        'price_category_label': pred_cat_label,
        'cluster': cluster_idx
    }

In [ ]:
# Example usage (you can adapt values to test):
# result = predict_for_product(
#     product_name='Wireless Bluetooth Headphones',
#     main_category='Electronics',
#     actual_price_clean=79.99,
#     discount_pct=20.0,
#     rating_clean=4.3,
#     rating_count_clean=5800,
#     popularity_score=0.75,
# )
# result

In [34]:
!python app.py

^C
